In [ ]:
!pip install timm coremltools # Cài luôn coremltools để tí nữa convert

import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import glob
import os

# --- CẤU HÌNH ĐƯỜNG DẪN ---
# Dựa trên screenshot của ông
IMG_DIR = '/kaggle/input/datasets/quocbao8925/coco-person-only/images/train/' 
BATCH_SIZE = 32
EPOCHS = 5 # Backbone chỉ cần 5-10 là đủ
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
!mkdir checkpoints
!gdown "https://drive.google.com/uc?id=1J6l8teyZrL0zFzHhzkC7efRhU0ZJ5G9Y&export=download&confirm=t" -O 'checkpoints/hmr2a.ckpt'

In [ ]:
import torch
import torch.nn as nn
from timm.models.vision_transformer import Block

class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=1280, depth=32, num_heads=16, ratio=4, **kwargs):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, ratio, qkv_bias=True)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return x[:, 0] # Chỉ lấy CLS token (feature 1280-dim)

In [ ]:
import torch.nn.functional as F

def load_hmr2_backbone_ultimate(checkpoint_path, device):
    model = ViT(img_size=224, patch_size=16, embed_dim=1280, depth=32, num_heads=16)
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['state_dict']
    
    new_state_dict = {}
    prefix = "backbone."
    
    # 1. Lọc và đổi tên cơ bản
    for k, v in state_dict.items():
        if k.startswith(prefix):
            clean_key = k[len(prefix):]
            # Sửa lỗi .proj của turn trước
            if 'patch_embed.proj.' in clean_key:
                clean_key = clean_key.replace('patch_embed.proj.', 'patch_embed.')
            # Sửa lỗi norm vs last_norm
            if clean_key == 'last_norm.weight': clean_key = 'norm.weight'
            if clean_key == 'last_norm.bias': clean_key = 'norm.bias'
            
            new_state_dict[clean_key] = v

    # 2. Xử lý thất lạc 'cls_token'
    if 'cls_token' not in new_state_dict:
        # Thử tìm xem nó có nằm ngoài prefix 'backbone.' không
        for k, v in state_dict.items():
            if 'cls_token' in k:
                new_state_dict['cls_token'] = v
                print(f">>> Đã tìm thấy 'cls_token' tại key: {k}")
                break

    # 3. Nội suy pos_embed (Giữ nguyên logic thành công của ông)
    if 'pos_embed' in new_state_dict:
        pos_embed_ckpt = new_state_dict['pos_embed']
        embedding_dim = pos_embed_ckpt.shape[-1]
        cls_token_embed = pos_embed_ckpt[:, :1, :]
        spatial_embed = pos_embed_ckpt[:, 1:, :] 
        
        # Nội suy từ 16x12 lên 14x14
        spatial_embed = spatial_embed.reshape(1, 16, 12, embedding_dim).permute(0, 3, 1, 2)
        spatial_embed = F.interpolate(spatial_embed, size=(14, 14), mode='bicubic', align_corners=False)
        spatial_embed = spatial_embed.permute(0, 2, 3, 1).reshape(1, 196, embedding_dim)
        
        new_state_dict['pos_embed'] = torch.cat((cls_token_embed, spatial_embed), dim=1)
        print(">>> Đã nội suy pos_embed thành công!")

    # 4. Nạp vào model
    msg = model.load_state_dict(new_state_dict, strict=False)
    print(f"Kết quả ghép não cuối cùng: {msg}")
    
    # Kiểm tra "sức khỏe" mô hình
    if not msg.missing_keys:
        print(">>> THẦY ĐÃ CÓ ĐỦ NÃO! Sẵn sàng Distill.")
    else:
        print(f">>> CẢNH BÁO: Vẫn thiếu các key: {msg.missing_keys}")

    model.to(device).eval()
    return model

teacher = load_hmr2_backbone_ultimate('/kaggle/working/checkpoints/hmr2a.ckpt', DEVICE)

In [ ]:
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset
import glob
import os
from PIL import Image

# 1. Thằng Trò FastViT - Tối ưu cho Apple Neural Engine
class FastViTStudent(nn.Module):
    def __init__(self, model_name='fastvit_sa24', target_dim=1280):
        super().__init__()
        # Load FastViT bản SA24 (cân bằng tốc độ/độ chính xác)
        # num_classes=0 để lấy feature trực tiếp
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        
        # Tự động lấy số chiều đầu ra của FastViT
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            student_out_dim = self.backbone(dummy_input).shape[1]
        
        # Lớp "đầu chuyển" để ép từ dimension của Student về 1280 của ViT-L
        self.proj = nn.Linear(student_out_dim, target_dim)

    def forward(self, x):
        features = self.backbone(x)
        return self.proj(features)

# 2. Bộ nạp dữ liệu (Dataset) - Chỉ quét ảnh, không cần nhãn
class DistillDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        # Chỉ lấy file .jpg trong thư mục train/images của ông
        self.img_paths = glob.glob(os.path.join(img_dir, '*.jpg'))
        self.transform = transform
        print(f"Đã nạp {len(self.img_paths)} ảnh để chuẩn bị vắt sữa.")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

import torchvision.transforms as T

# Bộ transform chuẩn để đưa ảnh về định dạng mà ViT-L và FastViT đều hiểu
transform = T.Compose([
    T.Resize((224, 224)), # Đưa về kích thước 224x224 chuẩn
    T.ToTensor(), # Chuyển ảnh thành Tensor
    # Chuẩn hóa theo thông số của bộ ImageNet (Thầy và Trò đều học từ đây)
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
from tqdm import tqdm
import torch

# --- KHỞI TẠO (Chỉ chạy 1 lần) ---
# Đảm bảo các biến EPOCHS, DEVICE, BATCH_SIZE đã được định nghĩa ở cell đầu
student = FastViTStudent().to(DEVICE)
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-4)
criterion = torch.nn.MSELoss()

dataset = DistillDataset(IMG_DIR, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# --- VÒNG LẶP LUYỆN CÔNG (Epoch Loop) ---
print(f"Bắt đầu quy trình vắt sữa feature từ Teacher sang Student trên {DEVICE}...")

for epoch in range(EPOCHS): # Thêm cái vòng lặp này vào thì mới có biến 'epoch'
    student.train()
    teacher.eval() # Thầy chỉ dạy, không học thêm (đóng băng)
    running_loss = 0.0
    
    # Bọc loader vào tqdm để hiện progress bar cho oai với đời
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}", unit="batch")
    
    for images in pbar:
        images = images.to(DEVICE)
        
        # 1. Thầy nhả feature (Không tính gradient cho nhẹ GPU)
        with torch.no_grad():
            t_feat = teacher(images) # Output: 1280 dim
            
        # 2. Trò nhái lại
        s_feat = student(images)
        loss = criterion(s_feat, t_feat)
        
        # 3. Cập nhật trọng số cho Trò
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Cập nhật loss realtime trên thanh bar
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_loss = running_loss / len(loader)
    print(f"Hoàn thành Epoch {epoch+1} - Loss trung bình: {avg_loss:.6f}")
    
    # --- LƯU LẠI THÀNH QUẢ SAU MỖI EPOCH ---
    checkpoint_path = f'fastvit_wham_epoch_{epoch+1}.pth'
    torch.save(student.state_dict(), checkpoint_path)
    print(f"Đã lưu 'linh đan' vào: {checkpoint_path}")

print("--- CHÚC MỪNG ÔNG GIÁO! LUYỆN CÔNG HOÀN TẤT ---")